In [15]:
import pandas as pd
from tqdm.std import tqdm
from openai import OpenAI
import re
import os
import json

In [16]:
input_data = "input_files/semantic_matching_0137.csv"
output_data = "output_files/semantic_matching_0137.xlsx"

In [ ]:
from dotenv import load_dotenv

load_dotenv()

True

## common functions

In [18]:
def to_pascal_case(phrase):
    # Remove parentheses and the content inside them, e.g., (Unaudited)
    phrase = re.sub(r'\s*\([^)]*\)', '', phrase)
    
    # Split on non-alphanumeric characters
    words = re.split(r'[^a-zA-Z0-9]', phrase)
    
    # Capitalize each word and join
    return ''.join(word.capitalize() for word in words if word)

In [19]:
def read_xml(xml_path):
    with open(xml_path, "r", encoding="utf-8") as f:
        content = f.read()

        return content

In [20]:
"""
    This function is different a bit among different tasks, here is for the semantic matching task
"""

def construct_annotation(df, results, task_name="semantic_matching"):
    """
        the options of task_name are semantic_matching, relationship_extraction, and mathematical_reasoning
    """
    new_df = df.copy(deep=True)

    schema = []
    presentation = []
    calculation = []
    definition = []
    label = []
    instance = []
    us_gaap = []
    true_answer = []
    segmentation_path = []
    parse_ok = []
    input = []
    outputByLLM = []
    
    for res in results:
        folder_name = res.get("folder_name")

        prefix_path = os.path.join(f"segmentation_filing/{task_name}", folder_name)
        real_segmentation_names = os.listdir(prefix_path)

        
        parse = False
        cal = None
        pre = None
        defn = None
        lab = None
        ins = None
        sch = None
        seg_path = None
        answers = [tag.get("dimension") for tag in res.get("error_tags")]
        for segmentation_name_ in real_segmentation_names:
            
            full_path = os.path.join(prefix_path, segmentation_name_)
    
            temp_cal = None
            temp_pre = None
            temp_defn = None
            temp_lab = None
            temp_ins = None
            temp_sch = None
            
            file_list = os.listdir(full_path)
            
            for file_name in file_list:
                if file_name.endswith("cal.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_cal = read_xml(final_path)
                elif file_name.endswith("pre.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_pre = read_xml(final_path)
                elif file_name.endswith("def.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_defn = read_xml(final_path)
                elif file_name.endswith("lab.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_lab = read_xml(final_path)
                elif file_name.endswith("htm.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_ins = read_xml(final_path)
                elif file_name.endswith("xsd"):
                    final_path = os.path.join(full_path,file_name)
                    temp_sch = read_xml(final_path)
                else:
                    continue

            is_appear = True

            ans = res.get("target_tag")
            # check the instance document
            if ans in temp_ins:
                pass
            elif ans.replace(":","") in temp_ins:
                pass
            elif ans.replace(":","_") in temp_ins:
                pass
            else:
                is_appear = False
           
            if is_appear:
                parse = True
                cal = temp_cal
                pre = temp_pre
                defn = temp_defn
                lab = temp_lab
                ins = temp_ins
                sch = temp_sch
                seg_path = os.path.join(folder_name, segmentation_name_)
                break
        
        calculation.append(cal)
        presentation.append(pre)
        definition.append(defn)
        label.append(lab)
        instance.append(ins)
        schema.append(sch)
        true_answer.append(answers)
        us_gaap.append(None)
        parse_ok.append(parse)
        segmentation_path.append(seg_path)
        input.append(None)
        outputByLLM.append(res)
     
        
    new_df["schema"] = schema
    new_df["presentation"] = presentation
    new_df["calculation"] = calculation
    new_df["definition"] = definition
    new_df["label"] = label
    new_df["instance"] = instance
    new_df["us_gaap"] = us_gaap
    new_df["input"] = input
    new_df["true_answer"] = true_answer
    new_df["segmentation_path"] = segmentation_path
    new_df["parse_successful"] = parse_ok
    new_df["LLM_Output"] = outputByLLM

    return new_df

## load dqc messages for semantic matching task

In [21]:
sm_137_df = pd.read_csv(input_data)

In [22]:
sm_137_df

,assertion.code,assertion.detail,assertion.run-date,assertion.severity,assertion.source,assertion.type,entity.cik,entity.name,report.accepted-timestamp,report.accession,report.base-taxonomy,report.creation-software,report.document-type,report.entry-url,report.filing-date,report.filing-year,report.id,report.sec-url,report.sic-code,folder_name
0,DQC.US.0137.9838,"2022-08-17 12:39:16,981 [DQC.US.0137.9838] The...",2022-08-17,ERROR,DQC,137,785956,J&J SNACK FOODS CORP.,2022-05-05 15:31:00,0001437749-22-011104,US GAAP 2022,Generated by ThunderDome Portal - 5/4/2022 10:...,10-Q,http://www.sec.gov/Archives/edgar/data/785956/...,2022-05-05,2022,469139,https://www.sec.gov/Archives/edgar/data/785956...,2052,10q-jjsf-20220326
1,DQC.US.0137.9838,"2022-08-17 12:39:16,982 [DQC.US.0137.9838] The...",2022-08-17,ERROR,DQC,137,785956,J&J SNACK FOODS CORP.,2022-05-05 15:31:00,0001437749-22-011104,US GAAP 2022,Generated by ThunderDome Portal - 5/4/2022 10:...,10-Q,http://www.sec.gov/Archives/edgar/data/785956/...,2022-05-05,2022,469139,https://www.sec.gov/Archives/edgar/data/785956...,2052,10q-jjsf-20220326
2,DQC.US.0137.9838,"2022-08-17 14:12:57,966 [DQC.US.0137.9838] The...",2022-08-17,ERROR,DQC,137,722313,NORTECH SYSTEMS INCORPORATED,2022-05-10 16:10:00,0001437749-22-011733,US GAAP 2022,Generated by ThunderDome Portal - 5/10/2022 4:...,10-Q,http://www.sec.gov/Archives/edgar/data/722313/...,2022-05-10,2022,471733,https://www.sec.gov/Archives/edgar/data/722313...,3679,10q-nsys-20220331
3,DQC.US.0137.9838,"2022-08-17 14:12:57,966 [DQC.US.0137.9838] The...",2022-08-17,ERROR,DQC,137,722313,NORTECH SYSTEMS INCORPORATED,2022-05-10 16:10:00,0001437749-22-011733,US GAAP 2022,Generated by ThunderDome Portal - 5/10/2022 4:...,10-Q,http://www.sec.gov/Archives/edgar/data/722313/...,2022-05-10,2022,471733,https://www.sec.gov/Archives/edgar/data/722313...,3679,10q-nsys-20220331
4,DQC.US.0137.9838,"2022-08-18 15:49:36,413 [DQC.US.0137.9838] The...",2022-08-18,ERROR,DQC,137,1376321,ZW Data Action Technologies Inc.,2022-05-16 16:06:00,0001171843-22-003730,US GAAP 2022,Generated by ThunderDome Portal - 5/13/2022 5:...,10-Q,http://www.sec.gov/Archives/edgar/data/1376321...,2022-05-16,2022,478178,https://www.sec.gov/Archives/edgar/data/137632...,7370,10q-cnet-20220331
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
120,DQC.US.0137.9838,"2024-05-16 18:57:45,141 [DQC.US.0137.9838] The...",2024-05-16,ERROR,DQC,137,875657,ULTRALIFE CORPORATION,2024-04-25 07:31:00,0001437749-24-013121,US GAAP 2024,Generated by ThunderDome XBRL - 10:26PM UTC 20...,10-Q,http://www.sec.gov/Archives/edgar/data/875657/...,2024-04-25,2024,703311,https://www.sec.gov/Archives/edgar/data/875657...,3690,10q-ulbi-20240331
121,DQC.US.0137.9838,"2024-05-16 18:57:45,142 [DQC.US.0137.9838] The...",2024-05-16,ERROR,DQC,137,875657,ULTRALIFE CORPORATION,2024-04-25 07:31:00,0001437749-24-013121,US GAAP 2024,Generated by ThunderDome XBRL - 10:26PM UTC 20...,10-Q,http://www.sec.gov/Archives/edgar/data/875657/...,2024-04-25,2024,703311,https://www.sec.gov/Archives/edgar/data/875657...,3690,10q-ulbi-20240331
122,DQC.US.0137.9838,"2024-05-16 18:57:45,142 [DQC.US.0137.9838] The...",2024-05-16,ERROR,DQC,137,875657,ULTRALIFE CORPORATION,2024-04-25 07:31:00,0001437749-24-013121,US GAAP 2024,Generated by ThunderDome XBRL - 10:26PM UTC 20...,10-Q,http://www.sec.gov/Archives/edgar/data/875657/...,2024-04-25,2024,703311,https://www.sec.gov/Archives/edgar/data/875657...,3690,10q-ulbi-20240331
123,DQC.US.0137.9838,"2024-05-16 18:57:45,142 [DQC.US.0137.9838] The...",2024-05-16,ERROR,DQC,137,875657,ULTRALIFE CORPORATION,2024-04-25 07:31:00,0001437749-24-013121,US GAAP 2024,Generated by ThunderDome XBRL - 10:26PM UTC 20...,10-Q,http://www.sec.gov/Archives/edgar/data/875657/...,2024-04-25,2024,703311,https://www.sec.gov/Archives/edgar/data/875657...,3690,10q-ulbi-20240331


## LLM judge assisstant

In [23]:
Client = OpenAI()

In [24]:
sm_137_system_prompt = """You are given a DQC validation message. Your task is to extract the following information:

1. Target Tag: Extract the primary us-gaap tag being referenced or discussed as erroneous in the message. Only include the tag explicitly stated as problematic or requiring correction.

2. Dimensions: Extract all dimension key-value pairs listed under "Dimensions" in the message. Each dimension should be in the format us-gaap:XXX=YYY.

3. For each extracted dimension, provide a short explanation in the "reason" field based on the message context. If the message does not explicitly mention an issue with the dimension, you may leave the "reason" field empty or give a general summary.

Output your result as a structured JSON in the following format:
```json
{
  "target_tag": "...",
  "error_tags": [
    {
      "dimension": "...",
      "reason": "..."
    }
  ]
}

"""

In [25]:
def get_response(dqc_message):
    
    completion = Client.chat.completions.create(
        model="gpt-4.1-mini",
        response_format={
            "type": "json_object"
        },
        messages=[
            {"role": "system", "content": sm_137_system_prompt},
            {"role": "user", "content": f"DQC validation message: {dqc_message} \n Output:"}
        ]
    )
    return completion.choices[0].message.content

In [26]:
res = []
for _, row in tqdm(sm_137_df.iterrows()):
    ## load each dqc message
    message = row["assertion.detail"]
    response = get_response(message)
    response = json.loads(response)

    ## load the folder name which will be used to obtain candidate segmentation folders
    folder_name = row["folder_name"]

    ## record the folder name
    response["folder_name"] = folder_name
    res.append(response)

125it [03:57,  1.90s/it]


In [27]:
res[4]

{'target_tag': 'us-gaap:OperatingLeaseRightOfUseAsset',
 'error_tags': [{'dimension': 'us-gaap:BalanceSheetLocationAxis=us-gaap:OtherCurrentAssetsMember',
   'reason': 'The BalanceSheetLocationAxis should not be used for this element; instead, the extensible enumeration element us-gaap:OperatingLeaseRightOfUseAssetStatementOfFinancialPositionExtensibleList should be used to indicate the location.'}],
 'folder_name': '10q-cnet-20220331'}

In [28]:
"""
    the options of task_name are semantic_matching, relationship_extraction, and mathematical_reasoning
"""
new_sm_137_df = construct_annotation(sm_137_df, res, task_name="semantic_matching")

In [29]:
new_sm_137_df

,assertion.code,assertion.detail,assertion.run-date,assertion.severity,assertion.source,assertion.type,entity.cik,entity.name,report.accepted-timestamp,report.accession,...,calculation,definition,label,instance,us_gaap,input,true_answer,segmentation_path,parse_successful,LLM_Output
0,DQC.US.0137.9838,"2022-08-17 12:39:16,981 [DQC.US.0137.9838] The...",2022-08-17,ERROR,DQC,137,785956,J&J SNACK FOODS CORP.,2022-05-05 15:31:00,0001437749-22-011104,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,[us-gaap:BalanceSheetLocationAxis=jjsf:Propert...,10q-jjsf-20220326/statement-note-13-leases-sup...,True,{'target_tag': 'us-gaap:FinanceLeaseRightOfUse...
1,DQC.US.0137.9838,"2022-08-17 12:39:16,982 [DQC.US.0137.9838] The...",2022-08-17,ERROR,DQC,137,785956,J&J SNACK FOODS CORP.,2022-05-05 15:31:00,0001437749-22-011104,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,[us-gaap:BalanceSheetLocationAxis=jjsf:Propert...,10q-jjsf-20220326/statement-note-13-leases-sup...,True,{'target_tag': 'us-gaap:FinanceLeaseRightOfUse...
2,DQC.US.0137.9838,"2022-08-17 14:12:57,966 [DQC.US.0137.9838] The...",2022-08-17,ERROR,DQC,137,722313,NORTECH SYSTEMS INCORPORATED,2022-05-10 16:10:00,0001437749-22-011733,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,[us-gaap:BalanceSheetLocationAxis=nsys:Propert...,10q-nsys-20220331/statement-note-5-leases-leas...,True,{'target_tag': 'us-gaap:FinanceLeaseRightOfUse...
3,DQC.US.0137.9838,"2022-08-17 14:12:57,966 [DQC.US.0137.9838] The...",2022-08-17,ERROR,DQC,137,722313,NORTECH SYSTEMS INCORPORATED,2022-05-10 16:10:00,0001437749-22-011733,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,[us-gaap:BalanceSheetLocationAxis=nsys:Propert...,10q-nsys-20220331/statement-note-5-leases-leas...,True,{'target_tag': 'us-gaap:FinanceLeaseRightOfUse...
4,DQC.US.0137.9838,"2022-08-18 15:49:36,413 [DQC.US.0137.9838] The...",2022-08-18,ERROR,DQC,137,1376321,ZW Data Action Technologies Inc.,2022-05-16 16:06:00,0001171843-22-003730,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,[us-gaap:BalanceSheetLocationAxis=us-gaap:Othe...,10q-cnet-20220331/statement-condensed-consolid...,True,{'target_tag': 'us-gaap:OperatingLeaseRightOfU...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
120,DQC.US.0137.9838,"2024-05-16 18:57:45,141 [DQC.US.0137.9838] The...",2024-05-16,ERROR,DQC,137,875657,ULTRALIFE CORPORATION,2024-04-25 07:31:00,0001437749-24-013121,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,[us-gaap:BalanceSheetLocationAxis=us-gaap:Othe...,10q-ulbi-20240331/statement-note-7-operating-l...,True,{'target_tag': 'us-gaap:OperatingLeaseRightOfU...
121,DQC.US.0137.9838,"2024-05-16 18:57:45,142 [DQC.US.0137.9838] The...",2024-05-16,ERROR,DQC,137,875657,ULTRALIFE CORPORATION,2024-04-25 07:31:00,0001437749-24-013121,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,[us-gaap:

In [30]:
new_sm_137_df.to_excel(output_data,index=False)